In [1]:
import os
import json
import duckdb
import pandas as pd
from dotenv import load_dotenv
import anthropic

pd.set_option("display.max_colwidth", 80)

NODES_PARQUET = "nodes_index.parquet"            # lean node lookup index (built below)
DISEASE_CATEGORIES = [                            # what counts as a disease/phenotype in KG2
    "biolink:Disease", "biolink:DiseaseOrPhenotypicFeature",
    "biolink:PhenotypicFeature", "biolink:PathologicalProcess",
]

con = duckdb.connect()
con.execute("SET enable_progress_bar=false")

load_dotenv()                                     # reads .env -> APIKEY
client = anthropic.Anthropic(api_key=os.environ["APIKEY"])
MODEL  = "claude-sonnet-4-5"

def llm(system, user, max_tokens=1024):
    msg = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        system=system, messages=[{"role": "user", "content": user}],
    )
    return msg.content[0].text

In [2]:
NODES_TSV = "rtx-kg2_nodes_2.8.1.tsv"   # headerless; cols: 4=description 7=id 10=name 13=synonym 15=category
READ_OPTS = ("delim='\t', header=false, quote='', escape='', ALL_VARCHAR=true, "
             "max_line_size=10000000, strict_mode=false, ignore_errors=true")

if not os.path.exists(NODES_PARQUET):
    print("Building node index (one full scan of the 4.9 GB nodes file)...")
    con.execute(f"""
        COPY (
            SELECT column07 AS id, column10 AS name, lower(column10) AS name_lc,
                   column15 AS category, column13 AS synonyms
            FROM read_csv('{NODES_TSV}', {READ_OPTS})
            WHERE column07 IS NOT NULL AND column10 IS NOT NULL
        ) TO '{NODES_PARQUET}' (FORMAT parquet)
    """)
n = con.execute(f"SELECT count(*) FROM '{NODES_PARQUET}'").fetchone()[0]
print(f"{n:,} nodes indexed -> {NODES_PARQUET}")

10,953,087 nodes indexed -> nodes_index.parquet
